In [1]:
import json
import re

In [2]:
TIME_RE = re.compile(r"([0-9]*\.?[0-9]*)\s*(s|ms|us|μs|ns)", re.IGNORECASE)

In [32]:
def get_data(input_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("JSON file must contain a list at the top level.")
    return data

def save_data(data, file):
    with open(file, "w", encoding="utf-8") as out:
        json.dump(data, out, indent=2, ensure_ascii=False)

def split_data(data, chunk_size):
    num_chunks = len(data) // chunk_size
    for i in range(0, len(data), chunk_size):
        chunk = data[i:i+chunk_size]
        output_file = f"results/results{i}.json"
        save_data(chunk, output_file)


In [28]:
def change_subtests(inputfiles):
    for file in inputfiles:
        data = get_data(file)
        for datapoint in data:
            for test in datapoint['tests']:
                test['tests'] = {
                    'vampire': {
                        'status': test['vampirestatus'],
                        'time': test['vampiretime'],
                    },
                    'prover': {
                        'status': test['proverstatus'],
                        'time': test['provertime'],
                    },
                }
                del test['vampirestatus']
                del test['vampiretime']
                del test['proverstatus']
                del test['provertime']
        save_data(data, file)


In [33]:
change_subtests([f'results/results{i}.json' for i in range(900, 901, 100)])

In [ ]:
def parse_times(data):
    for datapoint in data:
        for test in datapoint['tests']:
            try:
                test['provertime'] = parse_time(test['proverstatus'], test['provertime'])
                test['vampiretime'] = parse_time(test['vampirestatus'], test['vampiretime'])
            except KeyError as e:
                print(datapoint)
                raise e

def parse_time(status, time_str):
    if status == "Pending" or time_str is None:
        return None
    time_str = time_str.strip().lower()
    if time_str == "none" or time_str == "null":
        return None
    m = TIME_RE.fullmatch(time_str)
    if not m:
        raise ValueError(f"Invalid time format: {time_str}")
    num = float(m.group(1))
    unit = m.group(2).lower()
    if unit == "s":
        ms = num * 1000
    elif unit == "ms":
        ms = num
    elif unit == "us" or unit == "µs":
        ms = num / 1000
    elif unit == "ns":
        ms = num / 1_000_000
    else:
        raise ValueError(f"Unknown unit: {unit} cmp µs")
    if status == "Timedout":
        ms *= 5
    return ms


In [17]:
data = get_data("results/results0.json")
print(len(data))
print(data[0])

100
{'formula': '[](p->q)->[]p->[]q', 'setting': None, 'tests': [{'frames': 'K', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Theorem', 'provertime': 25.082}, {'frames': 'D', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Theorem', 'provertime': 1.303}, {'frames': 'T', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Theorem', 'provertime': 1.327}, {'frames': 'KB', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Pending', 'provertime': None}, {'frames': 'DB', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Pending', 'provertime': None}, {'frames': 'TB', 'vampirestatus': 'Theorem', 'vampiretime': 1.0, 'proverstatus': 'Pending', 'provertime': None}, {'frames': 'K4', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Pending', 'provertime': None}, {'frames': 'D4', 'vampirestatus': 'Theorem', 'vampiretime': 2.0, 'proverstatus': 'Pending', 'provertime': None}, {'frames': 'S4', 'vampirestatu

In [9]:
split_data(data, 100)